# Scenario A: reproducible plug-in classification

This notebook is a thin, paper-facing interface to the tested `mkv_classification` package. It fixes $D=2$ after the separate sensitivity study and evaluates the classifier only on independent final-test particle systems. Drift error is computed on additional independent auxiliary systems.

Set `PROFILE = "paper"` only when generating manuscript results. The smoke profile is an installation and integrity check and must not be reported.

## 1. Reproducible setup

All random streams, numerical settings, and sample-size meanings are stored in the configuration and output manifest. Here, every `N` is a particle count per class per interacting particle system.

In [ ]:
from __future__ import annotations
from pathlib import Path
import csv, sys
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'src').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from mkv_classification.config import main_profile
from mkv_classification.experiments import run_main_experiment

PROFILE = 'smoke'  # Change only this line to 'paper' for manuscript results.
config = main_profile(PROFILE)
output_dir = ROOT / 'results' / f'main_{PROFILE}'
config

## 2. Run or resume the experiment

Each completed `(regime, M, N, repetition)` is atomically checkpointed. Re-running this cell resumes rather than overwrites completed work.

In [ ]:
rows = run_main_experiment(config, output_dir)
print(f'{len(rows)} repetition-level rows available in {output_dir}.')

## 3. Paper-ready summary

The table reports the mean, sample standard deviation, standard error, and 95% interval across independent Monte Carlo repetitions.

In [ ]:
with (output_dir / 'summary.csv').open(newline='', encoding='utf-8') as handle:
    summary = list(csv.DictReader(handle))
columns = ['regime', 'M', 'N_train_per_class', 'repetitions',
           'plugin_accuracy_mean', 'plugin_accuracy_sample_standard_deviation',
           'plugin_accuracy_ci95_lower', 'plugin_accuracy_ci95_upper']
print(' | '.join(columns))
print(' | '.join(['---'] * len(columns)))
for row in summary:
    print(' | '.join(row[column] for column in columns))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8.2, 3.2), sharey=True)
for axis, regime in zip(axes, ['A(i)', 'A(ii)']):
    for M, marker in [(10, 'o'), (config.observation_steps[-1], 's')]:
        subset = [row for row in summary if row['regime'] == regime and int(row['M']) == M]
        subset.sort(key=lambda row: int(row['N_train_per_class']))
        if subset:
            axis.plot([int(row['N_train_per_class']) for row in subset],
                      [float(row['plugin_accuracy_mean']) for row in subset],
                      marker=marker, label=f'M={M}')
    axis.set_title(regime); axis.set_xlabel('Training particles per class'); axis.grid(alpha=.2)
axes[0].set_ylabel('Classification accuracy'); axes[-1].legend(frameon=False)
fig.tight_layout()
figure_path = output_dir / 'classification_accuracy.pdf'
fig.savefig(figure_path, bbox_inches='tight')
plt.show(); print(figure_path)

## 4. Reporting boundary

Only `paper` outputs may be copied into the manuscript. The fixed value $D=2$ is justified in `02_select_D.ipynb`; the final test systems used here never participate in that choice. The finite-particle true-drift comparison, when used, should be called an **oracle proxy**, not Bayes accuracy.